## MCO1 A Posteriori Analysis of String-Matching Algorithms

## Hypotheses
Null Hypothesis ( ): There is no significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the completion time of the string-matching process.

Alternative Hypothesis ( ): There is a significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on completion time. Specifically, as the Number of Queries increases to multiple simultaneous queries and Query Length increases, the Aho-Corasick algorithm will demonstrate a significantly lower completion time growth rate compared to the Naive and KMP algorithms.
 

Null Hypothesis ( ): There is no significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the total number of operations executed.

Alternative Hypothesis ( ): There is a significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the total number of operations. Specifically, as the Number of Queries and Query Length increase, the Aho-Corasick algorithm will execute significantly fewer total operations compared to the multiplicative operation growth seen in the Naive and KMP algorithms.



### Random Text Generation

##### Large Search String Generation (100000 characters)

In [ ]:
import random
import string

def generate_exact_random_text(length):
    """
    Generates a random string of an exact specified length.
    """
    character_pool = string.ascii_letters + string.digits + " "
    return ''.join(random.choices(character_pool, k=length))

# Generate the 100,000 character text
random_text_stress_test = generate_exact_random_text(100000)

# Verify the length of the generated text
print(f"Stress test text length: {len(random_text_stress_test):,} characters")

##### Query String Generation

In [1]:
import random
import string

# Generate strings within a range.
def generate_random_text(min_len, max_len):
    length = ""
    if (min_len< 100): length = "small"
    elif (min_len < 1000): length = "medium"
    else : length = "large"

    # Determine the random length within the specified boundary
    text_length = random.randint(min_len, max_len)
    
    # Use ASCII letters, digits, and spaces for the character pool
    character_pool = string.ascii_letters + string.digits + " "
    text =''.join(random.choices(character_pool, k=text_length))
    # Generate and return the random string
    return text , length



### Narrative Text Extraction

In [6]:
import random

def extract_narrative_text(file_path, min_len, max_len):
    """
    Reads a text file and extracts a random contiguous substring 
    with a length between min_len and max_len.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        full_text = file.read()
        
    # Isolate the actual story by stripping the Project Gutenberg header and footer
    start_marker = "CHAPTER I."
    end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
    
    start_idx = full_text.find(start_marker)
    end_idx = full_text.find(end_marker)
    
    # Fallback to the whole text if markers are not found
    if start_idx != -1 and end_idx != -1:
        story_text = full_text[start_idx:end_idx]
    else:
        story_text = full_text

    # Determine random length and valid starting position
    text_length = random.randint(min_len, max_len)
    max_start_pos = len(story_text) - text_length
    
    random_start = random.randint(0, max_start_pos)
    
    return story_text[random_start : random_start + text_length]

# Define the file name as provided
file_name = "aliceinwonderland.txt"

# Extract the three narrative text length categories
narrative_text_short = extract_narrative_text(file_name, 10, 100)
narrative_text_medium = extract_narrative_text(file_name, 100, 1000)
narrative_text_long = extract_narrative_text(file_name, 1000, 10000)

# Verify the lengths of the extracted texts
print(f"Narrative Short text length: {len(narrative_text_short)} characters")
print(f"Narrative Medium text length: {len(narrative_text_medium)} characters")
print(f"Narrative Long text length: {len(narrative_text_long)} characters")

Narrative Short text length: 89 characters
Narrative Medium text length: 915 characters
Narrative Long text length: 7292 characters


# Algorithm Implementation

### Naive/Brute-Force Algorithm

In [2]:
def naive_search(text, queries):
    """
    Executes the Naive/Brute-Force string-matching algorithm.
    Accepts a single string or a list of query strings.
    Returns the match indices and the total number of character comparisons.
    """
    # Standardize input to a list to handle both Single and Multiple IV conditions
    if isinstance(queries, str):
        queries = [queries]
        
    results = {query: [] for query in queries}
    total_comparisons = 0                       # Modification to calculate comparisons
    n = len(text)
    
    # Outer loop: Process each query sequentially
    for query in queries:
        m = len(query)
        if m == 0 or m > n:
            continue
            
        # Text scanning loop: Stop when remaining text is shorter than the query
        for i in range(n - m + 1):
            match = True
            
            # Character comparison loop
            for j in range(m):
                total_comparisons += 1          # Modification to calculate comparisons
                if text[i + j] != query[j]:
                    match = False
                    break  # Mismatch found, shift text pointer by 1
            
            if match:
                results[query].append(i)
                
    return results, total_comparisons



### Knuth-Morris-Pratt (KMP) Algorithm

In [3]:
def kmp_search(text, queries):
    """
    Executes the Knuth-Morris-Pratt (KMP) string-matching algorithm.
    Accepts a single string or a list of query strings.
    Returns the match indices and the total number of character comparisons.
    """
    # Standardize input to a list to handle both Single and Multiple IV conditions
    if isinstance(queries, str):
        queries = [queries]
        
    results = {query: [] for query in queries}
    total_comparisons = 0
    n = len(text)
    
    # Outer loop: Process each query sequentially
    for query in queries:
        m = len(query)
        if m == 0 or m > n:
            continue
            
        # Step 1: Preprocessing Phase - Build the Longest Prefix Suffix (LPS) array
        lps = [0] * m
        length = 0
        i = 1
        
        while i < m:
            total_comparisons += 1
            if query[i] == query[length]:
                length += 1
                lps[i] = length
                i += 1
            else:
                if length != 0:
                    length = lps[length - 1] # Fall back in the LPS array
                else:
                    lps[i] = 0
                    i += 1
                    
        # Step 2: Execution Phase - Scan the text using the LPS array
        i = 0  # text index
        j = 0  # query index
        
        while i < n:
            total_comparisons += 1
            if query[j] == text[i]:
                i += 1
                j += 1
                
                if j == m:
                    # Match found, record index and use LPS to avoid backtracking
                    results[query].append(i - j)
                    j = lps[j - 1]
            else:
                if j != 0:
                    j = lps[j - 1] # Skip matching characters using the prefix table
                else:
                    i += 1
                    
    return results, total_comparisons



### Aho-Corasick Algorithm

In [4]:
from collections import deque

class AhoCorasick:
    def __init__(self):
        # Initialize the trie with a root node (index 0)
        self.trie = [{'children': {}, 'fail': 0, 'outputs': []}]
        self.total_operations = 0

    def add_word(self, word):
        """Step 1: Insert a query string into the Trie."""
        current_node = 0
        for char in word:
            self.total_operations += 1
            if char not in self.trie[current_node]['children']:
                # Create a new node
                self.trie.append({'children': {}, 'fail': 0, 'outputs': []})
                self.trie[current_node]['children'][char] = len(self.trie) - 1
            current_node = self.trie[current_node]['children'][char]
        self.trie[current_node]['outputs'].append(word)

    def build_automaton(self):
        """Step 2: Build failure links using Breadth-First Search (BFS)."""
        queue = deque()
        
        # Initialize failure links for depth 1 nodes to root (0)
        for char, child_node in self.trie[0]['children'].items():
            self.trie[child_node]['fail'] = 0
            queue.append(child_node)
            self.total_operations += 1

        while queue:
            current_node = queue.popleft()
            
            for char, child_node in self.trie[current_node]['children'].items():
                queue.append(child_node)
                fail_node = self.trie[current_node]['fail']
                self.total_operations += 1

                # Follow failure links until a matching character or root is found
                while fail_node != 0 and char not in self.trie[fail_node]['children']:
                    self.total_operations += 1
                    fail_node = self.trie[fail_node]['fail']

                if char in self.trie[fail_node]['children']:
                    self.total_operations += 1
                    self.trie[child_node]['fail'] = self.trie[fail_node]['children'][char]
                else:
                    self.trie[child_node]['fail'] = 0

                # Merge outputs to catch overlapping subset matches 
                # (e.g., finding "he" when matching "she")
                self.trie[child_node]['outputs'].extend(
                    self.trie[self.trie[child_node]['fail']]['outputs']
                )

    def search(self, text, queries):
        """Step 3: Scan the text in a single pass."""
        for q in queries:
            self.add_word(q)
        self.build_automaton()

        results = {q: [] for q in queries}
        current_node = 0

        for i, char in enumerate(text):
            self.total_operations += 1
            
            # Follow failure links on mismatch
            while current_node != 0 and char not in self.trie[current_node]['children']:
                self.total_operations += 1
                current_node = self.trie[current_node]['fail']

            if char in self.trie[current_node]['children']:
                self.total_operations += 1
                current_node = self.trie[current_node]['children'][char]
            else:
                current_node = 0

            # Record any matches found at the current state
            for word in self.trie[current_node]['outputs']:
                # i is the end index of the match; calculate the start index
                results[word].append(i - len(word) + 1)

        return results, self.total_operations

def aho_corasick_search(text, queries):
    """
    Wrapper function to maintain the same API as naive_search and kmp_search.
    """
    if isinstance(queries, str):
        queries = [queries]
    
    ac = AhoCorasick()
    return ac.search(text, queries)


# Testing Section

In [5]:
# Global Variables
QUERY_INDEX = -1 

In [ ]:
def singleQuery(min, max):
    global QUERY_INDEX
    queries=[]
    query_length = ""
    if (min< 5): query_length = "small"
    elif (min < 10): query_length = "medium"
    else : query_length = "large"
    
    for i in range(30):
        queries.append(generate_random_text(min,max))

    QUERY_INDEX+=1
    return queries, query_length, "single" 

def multipleQuery(min, max, query_count):
    for i in range(30):
        queries.append([
            generate_random_text(min, max)
            for j in range(query_count)
        ])
    return queries

#### Random String Testing


| Length | elapsed_time | operations_count |
| :---     | :---:    | ---:     |
| 1-5     | ?   | ?   |
| 5-10    | ?    | ?   |
| 10-20    | ?    | ?   |

In [ ]:
import time
import polars as pl

# Naive


text_small, text_length = generate_random_text(10,100)
single_small_queries, query_length, query_type = singleQuery(1, 5)


rows = []
for query in single_small_queries:
    start_time= time.perf_counter_ns()
    single_results, single_ops = naive_search(text_small, query)
    end_time = time.perf_counter_ns()

    rows.append({
        "query_index": QUERY_INDEX,
        "text_length": text_length,
        "query_length": query_length, 
        "query_type": query_type,
        "query_count": "1",
        "success_flag":1 if next(iter(single_results.values())) else 0, 
        "elapsed_time (ns)": end_time-start_time,
        "operations_count": single_ops,

    })

df =pl.DataFrame(rows)
# print(df.head(2))

df.write_csv("single_ops_random")
# time_start_multi = time.perf_counter_ns()
# multi_results, multi_ops = naive_search(random_text_stress_test, multiple_queries)
# time_end_multi = time.perf_counter_ns()


# print(f"Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")

In [7]:
import time
import random
import string
import polars as pl

# 1. Pre-load narrative text once to avoid disk I/O overhead during the 3,240 runs
with open("aliceinwonderland.txt", "r", encoding="utf-8") as f:
    full_raw_text = f.read()

start_marker = "CHAPTER I."
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
s_idx = full_raw_text.find(start_marker)
e_idx = full_raw_text.find(end_marker)
ALICE_STORY = full_raw_text[s_idx:e_idx] if (s_idx != -1 and e_idx != -1) else full_raw_text

CHAR_POOL = string.ascii_letters + string.digits + " "

def generate_pure_string(min_len, max_len, text_type="random", source_text=None):
    """
    Generates a pure string (not a tuple) of length between min_len and max_len.
    For 'narrative', slices a contiguous substring from Alice in Wonderland
    or from the active trial's text so queries have realistic character distributions.
    """
    length = random.randint(min_len, max_len)
    if text_type == "random":
        return "".join(random.choices(CHAR_POOL, k=length))
    else:
        pool = source_text if (source_text and len(source_text) >= length) else ALICE_STORY
        start_pos = random.randint(0, len(pool) - length)
        return pool[start_pos : start_pos + length]

# 2. Define the experimental grid (2 x 3 x 3 x 2 x 3 = 108 combinations)
ALGORITHMS = {
    "Naive": naive_search,
    "KMP": kmp_search,
    "Aho-Corasick": aho_corasick_search,
}

TEXT_TYPES = ["random", "narrative"]

TEXT_LENGTH_SPECS = [
    ("small", 10, 100),
    ("medium", 100, 1000),
    ("large", 1000, 10000),
]

QUERY_LENGTH_SPECS = [
    ("small", 1, 5),
    ("medium", 5, 10),
    ("large", 10, 20),
]

# "Single" (1 query) vs "Multiple" (>= 2 simultaneous queries; e.g., 5 queries)
QUERY_TYPE_SPECS = [
    ("single", 1),
    ("multiple", 5),  # Adjust count (>= 2) or expand list to [2, 5, 10] if desired
]

NUM_TRIALS = 30

# 3. Exhaustive Execution Loop
rows = []
QUERY_INDEX = -1

for text_type in TEXT_TYPES:
    for text_len_label, t_min, t_max in TEXT_LENGTH_SPECS:
        for query_len_label, q_min, q_max in QUERY_LENGTH_SPECS:
            for query_type_label, q_count in QUERY_TYPE_SPECS:
                
                # Run 30 trials per condition combination
                for trial in range(NUM_TRIALS):
                    QUERY_INDEX += 1
                    
                    # Generate target text for this specific trial
                    target_text = generate_pure_string(t_min, t_max, text_type=text_type)
                    
                    # Generate query string(s) for this specific trial
                    queries = [
                        generate_pure_string(q_min, q_max, text_type=text_type, source_text=ALICE_STORY)
                        for _ in range(q_count)
                    ]
                    
                    # Test all 3 algorithms on the EXACT same (target_text, queries) pair
                    for algo_name, algo_func in ALGORITHMS.items():
                        start_time = time.perf_counter_ns()
                        results, ops_count = algo_func(target_text, queries)
                        end_time = time.perf_counter_ns()
                        
                        # Check if at least one query matched at least once
                        has_match = 1 if any(len(indices) > 0 for indices in results.values()) else 0
                        
                        rows.append({
                            "query_index": QUERY_INDEX,
                            "algorithm": algo_name,
                            "text_type": text_type,
                            "text_length": text_len_label,
                            "query_length": query_len_label,
                            "query_type": query_type_label,
                            "query_count": str(q_count),
                            "success_flag": has_match,
                            "elapsed_time (ns)": end_time - start_time,
                            "operations_count": ops_count,
                        })

# 4. Compile into Polars DataFrame and Export
df = pl.DataFrame(rows)
print(f"Total benchmark rows compiled: {df.height:,} (Expected: 108 * 30 = 3,240)")
print(df.head(6))

df.write_csv("string_matching_benchmark_results.csv")

Total benchmark rows compiled: 3,240 (Expected: 108 * 30 = 3,240)
shape: (6, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ query_ind ┆ algorithm ┆ text_type ┆ text_leng ┆ … ┆ query_cou ┆ success_f ┆ elapsed_t ┆ operatio │
│ ex        ┆ ---       ┆ ---       ┆ th        ┆   ┆ nt        ┆ lag       ┆ ime (ns)  ┆ ns_count │
│ ---       ┆ str       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ i64       ┆           ┆           ┆ str       ┆   ┆ str       ┆ i64       ┆ i64       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0         ┆ Naive     ┆ random    ┆ small     ┆ … ┆ 1         ┆ 0         ┆ 8900      ┆ 61       │
│ 0         ┆ KMP       ┆ random    ┆ small     ┆ … ┆ 1         ┆ 0         ┆ 6000      ┆ 63       │
│ 0         ┆ Aho-Coras ┆ random    ┆ small     ┆ … ┆ 1         ┆ 0         ┆ 19000     ┆ 67       │
│         

In [9]:
import statsmodels.formula.api as smf

# Step 5: Compute Descriptive Statistics (Mean and SD) across all 108 combinations in Polars
summary_df = (
    df.group_by(["algorithm", "text_type", "text_length", "query_length", "query_type"])
    .agg([
        pl.len().alias("n_trials"),
        pl.col("elapsed_time (ns)").mean().alias("mean_time_ns"),
        pl.col("elapsed_time (ns)").std().alias("sd_time_ns"),
        pl.col("operations_count").mean().alias("mean_ops"),
        pl.col("operations_count").std().alias("sd_ops"),
        pl.col("success_flag").mean().alias("match_rate"),
    ])
    .sort(["algorithm", "text_type", "text_length", "query_length", "query_type"])
)

print(summary_df.head(10))
summary_df.write_csv("descriptive_statistics_108_combinations.csv")

# Step 6: Fit Linear Mixed Models (LMM) for your two Hypotheses
# Convert Polars DataFrame to Pandas for statsmodels compatibility and rename the time column for formula parsing
pdf = df.rename({"elapsed_time (ns)": "elapsed_time_ns"}).to_pandas()

# Model 1: Completion Time (DV) ~ Algorithm * Query_Length * Query_Type + (1 | query_index)
# Grouping by query_index accounts for random trial-to-trial variance across the shared text/query pairs
lmm_time = smf.mixedlm(
    "elapsed_time_ns ~ C(algorithm) * C(query_length) * C(query_type)",
    data=pdf,
    groups=pdf["query_index"]
).fit()

print("=== LMM Results: Completion Time (ns) ===")
print(lmm_time.summary())

# Model 2: Operations Count (DV) ~ Algorithm * Query_Length * Query_Type + (1 | query_index)
lmm_ops = smf.mixedlm(
    "operations_count ~ C(algorithm) * C(query_length) * C(query_type)",
    data=pdf,
    groups=pdf["query_index"]
).fit()

print("\n=== LMM Results: Total Operations Count ===")
print(lmm_ops.summary())

shape: (10, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ algorithm ┆ text_type ┆ text_leng ┆ query_len ┆ … ┆ sd_time_n ┆ mean_ops  ┆ sd_ops    ┆ match_ra │
│ ---       ┆ ---       ┆ th        ┆ gth       ┆   ┆ s         ┆ ---       ┆ ---       ┆ te       │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆ str       ┆ str       ┆   ┆ f64       ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Aho-Coras ┆ narrative ┆ large     ┆ large     ┆ … ┆ 331988.93 ┆ 8608.8666 ┆ 3862.0794 ┆ 0.2      │
│ ick       ┆           ┆           ┆           ┆   ┆ 7999      ┆ 67        ┆           ┆          │
│ Aho-Coras ┆ narrative ┆ large     ┆ large     ┆ … ┆ 260040.16 ┆ 5673.8666 ┆ 2882.1902 ┆ 0.033333 │
│ ick       ┆           ┆           ┆           ┆   ┆ 2592      ┆ 67       

In [ ]:
import time
# KMP

# Example execution matching your project parameters
single_query = [generate_random_text(1,10)]
multiple_queries = [generate_random_text(1,10),generate_random_text(1,10),generate_random_text(1,10), generate_random_text(1,10)]


time_start_single = time.perf_counter_ns()
single_results, single_ops = kmp_search(random_text_stress_test, single_query)
time_end_single = time.perf_counter_ns()

time_start_multi = time.perf_counter_ns()
multi_results, multi_ops = kmp_search(random_text_stress_test, multiple_queries)
time_end_multi = time.perf_counter_ns()

print(f"KMP Single Query Operations: {single_ops}\tElapsed Time: {time_end_single-time_start_single} ns")
print(f"KMP Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")

In [ ]:
# Aho-Corasick

# Example execution matching your project parameters

single_query = [generate_random_text(1,10)]
multiple_queries = [generate_random_text(1,10),generate_random_text(1,10),generate_random_text(1,10), generate_random_text(1,10)]


time_start_single = time.perf_counter_ns()
single_results, single_ops = aho_corasick_search(random_text_stress_test, single_query)
time_end_single = time.perf_counter_ns()

time_start_multi = time.perf_counter_ns()
multi_results, multi_ops = aho_corasick_search(random_text_stress_test, multiple_queries)
time_end_multi = time.perf_counter_ns()

print(f"Aho-Corasick Single Query Operations: {single_ops}\tElapsed Time: {time_end_single-time_start_single} ns")
print(f"Aho-Corasick Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")

#### Narrative String Testing

In [ ]:
import time

# Naive

sample_text = "The quick brown fox jumps over the lazy dog"
single_query = ["fox"]
multiple_queries = ["the", "fox", "cat", "dog"]

time_start_single = time.perf_counter_ns()
single_results, single_ops = naive_search(sample_text, single_query)

time_end_single = time.perf_counter_ns()

time_start_multi = time.perf_counter_ns()
multi_results, multi_ops = naive_search(sample_text, multiple_queries)
time_end_multi = time.perf_counter_ns()

print(f"Single Query Operations: {single_ops}\tElapsed Time: {time_end_single-time_start_single} ns")

print(f"Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")

In [ ]:
import time
# KMP

# Example execution matching your project parameters
sample_text = "The quick brown fox jumps over the lazy dog"
single_query = ["fox"]
multiple_queries = ["the", "fox", "cat"]

time_start_single = time.perf_counter_ns()
single_results, single_ops = kmp_search(sample_text, single_query)
time_end_single = time.perf_counter_ns()

time_start_multi = time.perf_counter_ns()
multi_results, multi_ops = kmp_search(sample_text, multiple_queries)
time_end_multi = time.perf_counter_ns()

print(f"KMP Single Query Operations: {single_ops}\tElapsed Time: {time_end_single-time_start_single} ns")
print(f"KMP Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")

In [ ]:
# Aho-Corasick

# Example execution matching your project parameters
sample_text = "The quick brown fox jumps over the lazy dog"
single_query = ["fox"]
multiple_queries = ["the", "fox", "cat"]

time_start_single = time.perf_counter_ns()
single_results, single_ops = aho_corasick_search(sample_text, single_query)
time_end_single = time.perf_counter_ns()

time_start_multi = time.perf_counter_ns()
multi_results, multi_ops = aho_corasick_search(sample_text, multiple_queries)
time_end_multi = time.perf_counter_ns()

print(f"Aho-Corasick Single Query Operations: {single_ops}\tElapsed Time: {time_end_single-time_start_single} ns")
print(f"Aho-Corasick Multiple Query Operations: {multi_ops}\tElapsed Time: {(time_end_multi-time_start_multi)} ns")